In [3]:
# 数据处理
%pip install pandas numpy jieba scikit-learn joblib
import pandas as pd
import numpy as np
import re

# 中文分词
import jieba

# 机器学习
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

# 模型保存
import joblib

# 读取数据集
df = pd.read_csv(r"E:\graduation project\E-commerce Review Sentiment Classification\data\online_shopping_10_cats.csv")  
                 
# 查看原始数据基本信息
print(f"原始数据总量：{len(df)}")
print(f"原始标签分布：\n{df['label'].value_counts()}")  # label是情感标签列名 1正面 0负面
print(f"{df['cat'].value_counts()}")

# 保留非书籍评论
df = df[df['cat'] != '书籍']

Note: you may need to restart the kernel to use updated packages.
原始数据总量：62774
原始标签分布：
label
1    31728
0    31046
Name: count, dtype: int64
cat
平板     10000
水果     10000
洗发水    10000
衣服     10000
酒店     10000
计算机     3992
书籍      3851
手机      2323
蒙牛      2033
热水器      575
Name: count, dtype: int64



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# 定义清洗函数：过滤空文本、重复、广告、短文本

def clean_data(df, review_col="review", label_col="label"):
    # 1. 过滤空文本
    df = df[df[review_col].notna() & (df[review_col] != "")]
    # 2. 去重（删除重复评论）
    df = df.drop_duplicates(subset=[review_col])
    # 3. 过滤广告/无关文本（比如“加微信”“领券”“刷单”）
    ad_pattern = r"加微信|领券|刷单|代购|红包|二维码|链接"
    df = df[~df[review_col].str.contains(ad_pattern, regex=True)]
    # 4. 过滤过短文本（比如少于3个字的无意义评论）
    df = df[df[review_col].str.len() >= 3]
    # 5. 重置索引
    df = df.reset_index(drop=True)
    return df

# 执行清洗
df = clean_data(df, review_col="review", label_col="label")
print(f"清洗后数据量：{len(df)}")
print(f"清洗后标签分布：\n{df['label'].value_counts()}")
print(f"{df['cat'].value_counts()}")

清洗后数据量：58767
清洗后标签分布：
label
1    29565
0    29202
Name: count, dtype: int64
cat
酒店     9997
水果     9984
平板     9978
洗发水    9959
衣服     9957
计算机    3980
手机     2317
蒙牛     2032
热水器     563
Name: count, dtype: int64


In [5]:
# 加载中文停用词

def load_stopwords(stopword_file=r"E:\graduation project\E-commerce Review Sentiment Classification\data\stopwords.txt"):
    stopwords = set()
    
    try:
        # 打开文件，按行读取，编码兼容UTF-8和GBK（常见的中文编码）
        with open(stopword_file, "r", encoding="utf-8") as f:
            lines = f.readlines()
        
        # 处理每一行：去除换行符、空格，空行不加入
        for line in lines:
            word = line.strip()  # 去除首尾的空格、换行符
            if word:  # 跳过空行
                stopwords.add(word)
        
        if len(stopwords) == 0:
            print("警告：停用词文件为空，请检查文件内容")
        else:
            print(f"停用词加载完成，共 {len(stopwords)} 个")
    
    except FileNotFoundError:
        print(f"错误：未找到停用词文件 {stopword_file}，请检查文件路径")
    except Exception as e:
        print(f"加载停用词失败：{str(e)}")
    
    return stopwords

# 调用函数加载停用词
STOPWORDS = load_stopwords()

停用词加载完成，共 63 个


In [6]:
# 定义文本清洗函数

def clean_text(text):
    if pd.isna(text):
        return ''
    
    text = str(text)
    
    # 去除各类括号
    text = re.sub(r'[【】\[\]（）()《》<>「」『』【】〔〕]', '', text)
    
    # 去除英文和数字
    text = re.sub(r'[a-zA-Z0-9]', '', text)
    
    # 去除特殊符号
    text = re.sub(r'[▼◆▶■●★☆◆◇○◎●△▲※→←↑↓]', '', text)
    
    # 去除空白字符
    text = re.sub(r'\s+', '', text)
    
    return text

# 测试清洗效果
test_text = "【测试】这个产品很好！！★★★★★ 100分！"
print(f"原始文本: {test_text}")
print(f"清洗后: {clean_text(test_text)}")

原始文本: 【测试】这个产品很好！！★★★★★ 100分！
清洗后: 测试这个产品很好！！分！


In [7]:
# 定义分词函数

def cut_words(text):
   
    # 结巴分词
    words = jieba.lcut(text)
    
    # 去除停用词和长度<2的词
    words = [w for w in words if w not in STOPWORDS and len(w) >= 2]
    
    return ' '.join(words)

# 测试分词效果
test_clean = clean_text("这个产品真的很不错，物流也很快！")
print(f"清洗后: {test_clean}")
print(f"分词后: {cut_words(test_clean)}")

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\ljh01307\AppData\Local\Temp\jieba.cache


清洗后: 这个产品真的很不错，物流也很快！


Loading model cost 0.490 seconds.
Prefix dict has been built successfully.


分词后: 产品 不错 物流 很快


In [8]:
# 对全部评论进行预处理

def preprocess_reviews(df, text_col='review'):

    print("开始文本预处理...")
    
    # 清洗文本
    df['clean_review'] = df[text_col].apply(clean_text)
    print("清洗完成")
    
    # 分词
    df['seg_review'] = df['clean_review'].apply(cut_words)
    print("分词完成")
    
    # 统计信息
    avg_words = df['seg_review'].str.split().str.len().mean()
    print(f"\n预处理完成！")
    print(f"平均每条评论词数: {avg_words:.1f}")
    print(f"空评论数量: {sum(df['seg_review'] == '')}")
    
    return df

# 执行预处理
df = preprocess_reviews(df)
df[['review', 'clean_review', 'seg_review']].head(3)

开始文本预处理...
清洗完成
分词完成

预处理完成！
平均每条评论词数: 14.7
空评论数量: 126


,review,clean_review,seg_review
0,﻿很不错。。。。。。很好的平板,﻿很不错。。。。。。很好的平板,不错 平板
1,帮同学买的，同学说感觉挺好，质量也不错,帮同学买的，同学说感觉挺好，质量也不错,同学 同学 质量 不错
2,东西不错，一看就是正品包装，还没有开机，相信京东，都是老顾客，还是京东值得信赖，给五星好评,东西不错，一看就是正品包装，还没有开机，相信京东，都是老顾客，还是京东值得信赖，给五星好评,东西 不错 一看 正品 包装 开机 相信 京东 顾客 京东 值得 信赖 五星 好评


In [9]:
# 划分训练集和测试集

# 提取特征和标签
X = df['seg_review'].values
Y = df['label'].values

print(f"总样本数: {len(X)}")
print(f"好评比例: {sum(Y==1)/len(Y):.2%}")
print(f"差评比例: {sum(Y==0)/len(Y):.2%}")

# 划分训练集和测试集（分层抽样）
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, 

    test_size=0.2, 
    random_state=666, 
    stratify=Y  # 保持类别比例
)

print(f"\n数据集划分完成")
print(f"训练集: {len(X_train)} 条")
print(f"测试集: {len(X_test)} 条")

总样本数: 58767
好评比例: 50.31%
差评比例: 49.69%

数据集划分完成
训练集: 47013 条
测试集: 11754 条


In [10]:
# TF-IDF特征提取

def extract_features(X_train, X_test, vectorizer_type='tfidf', max_features=8000):
    print(f"提取特征，最大特征数: {max_features}")
    
    if vectorizer_type == 'tfidf':
        vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),      # 一元+二元词组
            min_df=2,                # 至少在2个文档中出现
            max_df=0.8,              # 忽略在80%以上文档出现的词
            sublinear_tf=True,       # 使用1+log(tf)替代tf
            use_idf=True
        )
        print("使用TF-IDF向量化")
    else:
        vectorizer = CountVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.8
        )
        print("使用词频向量化")
    
    # 拟合并转换训练集
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)
    
    print(f"特征提取完成！")
    print(f"特征维度: {X_train_vec.shape[1]}")
    print(f"训练集特征矩阵形状: {X_train_vec.shape}")
    print(f"测试集特征矩阵形状: {X_test_vec.shape}")
    
    return X_train_vec, X_test_vec, vectorizer

# 执行特征提取
X_train_vec, X_test_vec, vectorizer = extract_features(
    X_train, X_test, 
    vectorizer_type='tfidf', 
    max_features=8000
)

提取特征，最大特征数: 8000
使用TF-IDF向量化
特征提取完成！
特征维度: 8000
训练集特征矩阵形状: (47013, 8000)
测试集特征矩阵形状: (11754, 8000)


In [11]:
# 训练多个模型并交叉验证

def train_models(X_train, Y_train):

    models = {
        'LogisticRegression': LogisticRegression(
            C=2.0, 
            max_iter=1000, 
            solver='liblinear', 
            random_state=42
        ),
        'NaiveBayes': MultinomialNB(
            alpha=0.1
        ),
        'LinearSVC': LinearSVC(
            C=1.0, 
            max_iter=2000, 
            random_state=42
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=100, 
            max_depth=20, 
            random_state=42, 
            n_jobs=-1
        ),
        # 'GBDT': GradientBoostingClassifier(
        #     n_estimators=80, 
        #     max_depth=5, 
        #     learning_rate=0.1, 
        #     random_state=42
        # )
    
    }
    
    results = {}
    
    print("="*60)
    print("开始5折交叉验证训练")
    print("="*60)
    
    for name, model in models.items():
        print(f"\n训练模型: {name}")
        
        # 训练模型
        model.fit(X_train, Y_train)
        
        # 5折交叉验证
        cv_scores = cross_val_score(model, X_train, Y_train, cv=5, scoring='f1')
        
        results[name] = {
            'model': model,
            'cv_mean': cv_scores.mean(),
            'cv_std': cv_scores.std()
        }
        
        print(f"CV F1分数: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")
    
    return results

# 执行多模型训练
model_results = train_models(X_train_vec, Y_train)

# 显示对比结果
print("\n" + "="*60)
print("模型对比结果")
print("="*60)
results_df = pd.DataFrame([
    {'模型': k, '平均F1': v['cv_mean'], '标准差': v['cv_std']} 
    for k, v in model_results.items()
]).sort_values('平均F1', ascending=False)

print(results_df.to_string(index=False))

开始5折交叉验证训练

训练模型: LogisticRegression
CV F1分数: 0.9061 (±0.0015)

训练模型: NaiveBayes
CV F1分数: 0.9007 (±0.0009)

训练模型: LinearSVC
CV F1分数: 0.8993 (±0.0015)

训练模型: RandomForest
CV F1分数: 0.8325 (±0.0037)

模型对比结果
                模型     平均F1      标准差
LogisticRegression 0.906094 0.001485
        NaiveBayes 0.900656 0.000917
         LinearSVC 0.899256 0.001525
      RandomForest 0.832474 0.003728


In [12]:
# 逻辑回归超参数网格搜索

def optimize_lr(X_train, Y_train):

    print("\n" + "="*60)
    print("逻辑回归超参数优化")
    print("="*60)
    
    param_grid = {
        'C': [0.1, 0.5, 1.0, 2.0, 5.0],
        'solver': ['liblinear', 'saga'],
        'penalty': ['l2']  # liblinear支持l2，saga支持l1/l2
    }
    
    lr = LogisticRegression(max_iter=1000, random_state=42)
    
    grid = GridSearchCV(
        lr, 
        param_grid, 
        cv=5, 
        scoring='f1',
        n_jobs=-1,
        verbose=1
    )
    
    print("正在搜索最佳参数...")
    grid.fit(X_train, Y_train)
    
    print(f"\n优化完成！")
    print(f"最佳参数: {grid.best_params_}")
    print(f"最佳F1分数: {grid.best_score_:.4f}")
    
    return grid.best_estimator_

# 执行优化
best_lr = optimize_lr(X_train_vec, Y_train)


逻辑回归超参数优化
正在搜索最佳参数...
Fitting 5 folds for each of 10 candidates, totalling 50 fits

优化完成！
最佳参数: {'C': 2.0, 'penalty': 'l2', 'solver': 'liblinear'}
最佳F1分数: 0.9061


d:\python\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [13]:
# SVM超参数网格搜索

def optimize_svc(X_train, Y_train):

    print("\n" + "="*60)
    print("线性SVM超参数优化")
    print("="*60)
    
    param_grid = {
        'C': [0.1, 0.5, 1.0, 2.0],
        'loss': ['hinge', 'squared_hinge'],
        'penalty': ['l2']
    }
    
    svc = LinearSVC(max_iter=2000, random_state=42)
    
    grid = GridSearchCV(
        svc, 
        param_grid, 
        cv=5, 
        scoring='f1',
        n_jobs=-1,
        verbose=1
    )
    
    print("正在搜索最佳参数...")
    grid.fit(X_train, Y_train)
    
    print(f"\n优化完成！")
    print(f"最佳参数: {grid.best_params_}")
    print(f"最佳F1分数: {grid.best_score_:.4f}")
    
    return grid.best_estimator_

# 执行优化
best_svc = optimize_svc(X_train_vec, Y_train)


线性SVM超参数优化
正在搜索最佳参数...
Fitting 5 folds for each of 8 candidates, totalling 40 fits

优化完成！
最佳参数: {'C': 0.1, 'loss': 'squared_hinge', 'penalty': 'l2'}
最佳F1分数: 0.9063


In [15]:
# 详细评估模型性能

def evaluate_model(model, X_test, Y_test, model_name='Model'):
    
    # 预测
    y_pred = model.predict(X_test)
    
    # 计算指标
    acc = accuracy_score(Y_test, y_pred)
    f1 = f1_score(Y_test, y_pred)
    
    print("\n" + "="*60)
    print(f"{model_name} 评估结果")
    print("="*60)
    print(f"准确率: {acc:.4f}")
    print(f"F1分数: {f1:.4f}")
    
    print("\n分类报告:")
    print(classification_report(Y_test, y_pred, target_names=['差评', '好评']))
    
    # 混淆矩阵
    cm = confusion_matrix(Y_test, y_pred)
    print("\n混淆矩阵:")
    print(f"          预测差评  预测好评")
    print(f"实际差评   {cm[0,0]:6d}   {cm[0,1]:6d}")
    print(f"实际好评   {cm[1,0]:6d}   {cm[1,1]:6d}")
    
    # 计算各类错误率
    tn, fp, fn, tp = cm.ravel()
    print(f"\n详细指标:")
    print(f"  特异度 (TNR): {tn/(tn+fp):.4f}")
    print(f"  灵敏度 (TPR): {tp/(tp+fn):.4f}")
    print(f"  精确率 (PPV): {tp/(tp+fp):.4f}")
    
    return acc, f1

# 评估优化后的模型
print("\n评估优化后的逻辑回归模型")
lr_acc, lr_f1 = evaluate_model(best_lr, X_test_vec, Y_test, "优化后逻辑回归")

print("\n评估优化后的SVM模型")
svc_acc, svc_f1 = evaluate_model(best_svc, X_test_vec, Y_test, "优化后线性SVM")


评估优化后的逻辑回归模型

优化后逻辑回归 评估结果
准确率: 0.9125
F1分数: 0.9128

分类报告:
              precision    recall  f1-score   support

          差评       0.91      0.91      0.91      5841
          好评       0.91      0.91      0.91      5913

    accuracy                           0.91     11754
   macro avg       0.91      0.91      0.91     11754
weighted avg       0.91      0.91      0.91     11754


混淆矩阵:
          预测差评  预测好评
实际差评     5340      501
实际好评      528     5385

详细指标:
  特异度 (TNR): 0.9142
  灵敏度 (TPR): 0.9107
  精确率 (PPV): 0.9149

评估优化后的SVM模型

优化后线性SVM 评估结果
准确率: 0.9132
F1分数: 0.9133

分类报告:
              precision    recall  f1-score   support

          差评       0.91      0.92      0.91      5841
          好评       0.92      0.91      0.91      5913

    accuracy                           0.91     11754
   macro avg       0.91      0.91      0.91     11754
weighted avg       0.91      0.91      0.91     11754


混淆矩阵:
          预测差评  预测好评
实际差评     5359      482
实际好评      538     5375

详细指标:
  特异

In [18]:
# 保存最佳模型
import os
# 创建保存目录
model_dir = r"E:\\graduation project\\E-commerce Review Sentiment Classification\\models"
os.makedirs(model_dir, exist_ok=True)

# 保存模型和向量化器
joblib.dump(best_svc, os.path.join(model_dir, "best_svm_model.pkl"))
joblib.dump(vectorizer, os.path.join(model_dir, "tfidf_vectorizer.pkl"))
joblib.dump(best_lr, os.path.join(model_dir, "best_lr_model.pkl"))

print(f"模型已保存至: {model_dir}")
print(f"SVM测试集F1分数: {svc_f1:.4f}")
print(f"逻辑回归测试集F1分数: {lr_f1:.4f}")

模型已保存至: E:\\graduation project\\E-commerce Review Sentiment Classification\\models
SVM测试集F1分数: 0.9133
逻辑回归测试集F1分数: 0.9128


In [19]:
# 单条评论预测函数

def predict_review(review_text, model, vectorizer):
    # 清洗
    cleaned = clean_text(review_text)
    # 分词
    seg_text = cut_words(cleaned)
    # 向量化
    vec = vectorizer.transform([seg_text])
    # 预测
    pred = model.predict(vec)[0]
    
    sentiment = "好评" if pred == 1 else "差评"
    return sentiment, seg_text

# 测试几个例子
test_reviews = [
    "这个产品质量很好，非常满意！",
    "东西太差了，用了一次就坏了",
    "性价比一般，没有想象中好",
    "什么狗屎玩意，狗都不用",
    "感觉还行啊兄弟们，可以冲"
]

print("="*60)
print("单条评论预测测试")
print("="*60)
for review in test_reviews:
    sentiment, seg = predict_review(review, best_svc, vectorizer)
    print(f"评论: {review[:15]}... -> 预测: {sentiment}")

单条评论预测测试
评论: 这个产品质量很好，非常满意！... -> 预测: 好评
评论: 东西太差了，用了一次就坏了... -> 预测: 差评
评论: 性价比一般，没有想象中好... -> 预测: 差评
评论: 什么狗屎玩意，狗都不用... -> 预测: 差评
评论: 感觉还行啊兄弟们，可以冲... -> 预测: 好评
